# Estatística Multivariada (Decodificação / MVPA) em Dados de MEG/EEG

Vamos começar a usar Machine Learning!

In [ ]:
# Medida de segurança caso as janelas interativas não estejam aparecendo

%matplotlib qt 

'''

%matplotlib qt:

- Troca o backend para Qt (como use('Qt5Agg') faria)
- Configura o Jupyter para abrir as figuras em janelas externas, e não dentro da célula.

Sem esse comando, mesmo usando Qt5Agg, o Jupyter pode tentar renderizar figuras dentro da própria página.

Então surge a dúvida: mas não já temos o matplotlib.use('Qt5Agg')

Esse comando troca o backend do Matplotlib no Python inteiro, dizendo:
- “Use o Qt5Agg como backend gráfico.” Ou seja, diz como o Matplotlib vai renderizar as figuras internamente.

Porém: isso não diz ao Jupyter Notebook onde exibir as figuras.


Estava funcionando nos outros notebooks sem o '%matplotlib qt'mas aparentemente é melhor colocar esse comando igualmente, para evitar bugs
'''

In [ ]:
import pathlib
import matplotlib
import matplotlib.pyplot as plt
import mne

matplotlib.use('Qt5Agg')
mne.set_log_level('warning')

## Carregue as epochs

Vamos apenas usar os dados pré processados, sem o ICA (apenas para testes)

In [ ]:
epochs = mne.read_epochs(pathlib.Path('out_data') / 'epochs_epo.fif')
epochs.apply_baseline((None, 0))
epochs

# Número de eventos/epochs:
    # Auditory/Left: 72
    # Auditory/Right: 73
    

## Selecionar epochs de interesse

Aqui, pretendemos analisar as epochs auditivas.

In [ ]:
epochs_auditory = epochs['Auditory']
epochs_auditory

## Calcular a 'evoked difference' empírica

In [ ]:
# Cria um Evoked de diferença entre dois Evokeds:
# combine_evoked recebe uma lista de Evokeds e aplica uma combinação linear.
# Aqui usamos pesos [1, -1], então o resultado é: Evoked_Left - Evoked_Right
evoked_diff = mne.combine_evoked(
    [
        epochs_auditory['Auditory/Left'].average(),   # Média dos epochs do estímulo auditivo esquerdo
        epochs_auditory['Auditory/Right'].average()   # Média dos epochs do estímulo auditivo direito
    ],
    weights=[1, -1]  # Define a operação: 1*Left + (-1)*Right → diferença entre condições
)

# Plota o Evoked de diferença.
# gfp=True adiciona o gráfico do Global Field Power, que mostra a força total do campo magnético no MEG ou atividade elétrica ao longo do tempo. ELe é calculado de modo correto em cima do evoked da diferença
evoked_diff.plot(gfp=True)

# Compara visualmente vários Evokeds em uma mesma figura (que é interativa):
# - Evoked Left
# - Evoked Right
# - Diferença Left - Right
mne.viz.plot_compare_evokeds(
    [
        epochs_auditory['Auditory/Left'].average(),   # Evoked do estímulo auditivo esquerdo
        epochs_auditory['Auditory/Right'].average(),  # Evoked do estímulo auditivo direito
        evoked_diff                                   # Diferença entre eles
    ]
)



<p>
  <a href="/">
    <img src="imgs/Cálculo de GFP.png" alt="GFP">
  </a>
</p>

Queremos mais que isso, vamos usar Machine Learning

## Equalize o número de epochs

>Para manter o nível de chance em 50% de acurácia, primeiro equalizamos o número de epochs em cada condição.


Idealmente o número de epochs por condição deve ser igual, especialmente quando você vai:

* **comparar condições** (estatística)
* **treinar um classificador** (machine learning)
* **gerar evokeds para comparação direta**
* **computar decoding time-resolved**
* **manter chance level correto (50%)**

Vamos por partes.



### Por que equalizar o número de epochs?

Se você tem:

* 300 epochs da condição A
* 100 epochs da condição B

E você compara A vs B, então:

* A terá um **Evoked mais limpo e mais estável**
* B terá um **Evoked mais ruidoso**
* A diferença entre condições pode refletir ruído, não efeito neural
* Classificadores podem ficar **enviesados** pela classe com mais exemplos

Isso é chamado **class imbalance**.



#### Para análise de decodificação (classificação)

Quando você usa métodos no MNE como:

* `mne.decoding.SlidingEstimator`
* `GeneralizingEstimator`
* qualquer classificador scikit-learn

Se uma classe tiver muito mais epochs:

* o modelo aprende a “chutar sempre a classe maior”
* o accuracy sobe artificialmente
* seu **chance level deixa de ser 50%**

É um viés estatístico clássico.

Por isso existe o método:

```python
epochs.equalize_event_counts()
```

ou manualmente:

```python
epochs_a, epochs_b = mne.epochs.equalize_epoch_counts([epochs_a, epochs_b])
```



#### Para Evoked (média dos epochs)

Um Evoked de 300 trials é mais limpo do que um Evoked de 100 trials.
Comparar os dois diretamente também pode ser injusto — a diferença pode vir só da diferença de ruído.

Então equalizar resolve isso:

* mantém ruído equivalente
* deixa as comparações simétricas
* reduz viés


### Então igualar epochs é **obrigatório?**

Depende:

| Situação                             | Precisa equalizar?    |
| ------------------------------------ | --------------------- |
| Comparar evokeds                     | **Recomendado**       |
| Decodificação / classificação        | **Quase obrigatório** |
| Estatísticas de diferença            | **Quase obrigatório** |
| Análises dentro de uma só condição   | Não precisa           |
| Modelagem de fonte (forward/inverse) | Não precisa           |
| ICA / pré-processamento              | Não precisa           |

In [ ]:
# Função que equaliza de modo automático

'''
epochs_auditory.event_id é um dicionário, por exemplo:
{'Auditory/Left': 1, 'Auditory/Right': 2}

O MNE então procura as condições dentro dos epochs e descarta epochs das condições que têm mais trials,
até que todas tenham a mesma quantidade.
'''
epochs_auditory.equalize_event_counts(epochs_auditory.event_id)
epochs_auditory

## Criar a entrada X e a resposta y.

Um classificador recebe como entrada uma matriz `X` e retorna um vetor `y` (composto por `0` e `1`). Aqui, `X` será **os dados em um único ponto no tempo de todos os gradiômetros** (daí o termo multivariado). Queremos treinar nosso modelo para discriminar entre os ensaios `Auditory/Left` e `Auditory/Right`.

Trabalhamos com todos os sensores conjuntamente e buscamos encontrar um padrão discriminativo entre as duas condições para prever a condição experimental de ensaios individuais.

Para a classificação, utilizaremos o pacote `scikit-learn` ([http://scikit-learn.org/](http://scikit-learn.org/)) e funções do MNE-Python.

Primeiro, vamos criar o vetor de resposta, `y`.


### Classifiers

No scikit-learn, todo classificador funciona assim:

* Você dá **X**, uma matriz com seus dados.
* Ele aprende padrões nesses dados.
* Ele tenta prever **y**, que são os rótulos (labels) das condições.

Formalmente:

```markdown
X → matriz de features (o que o modelo “vê”)
y → vetor de labels (o que o modelo deve prever)
```

#### O que é X no caso de MEG/EEG?


> X é o dado num ponto temporal em cada gradiômetro

Isso significa:

* Escolhe um **instante de tempo**, por exemplo **t = 100 ms**
* Pega a atividade de **todos os canais do tipo gradiômetro** nesse instante
* Forma um vetor com esses valores

Se você tem 203 gradiômetros, então X (para um trial) será algo como:

```
[0.1, -0.03, 0.04, 0.25, ..., -0.12]   → 203 valores
```

Para *todos* os trials juntos, vira uma matriz:

```
X.shape = (n_epochs, n_sensors)
```

Por exemplo:

```python
X = [ [trial1_sensor1, trial1_sensor2, ..., trial1_sensor203],
      [trial2_sensor1, trial2_sensor2, ..., trial2_sensor203],
      ...
    ]
```


#### O que é y?

O classificador precisa saber qual rótulo (condição) cada trial pertence.

Para o nosso caso:

* `Auditory/Left` → classe 0
* `Auditory/Right` → classe 1

Então o vetor y é algo como:

```python
y = [0, 0, 0, 0, 1, 1, 1, 0, ...]
```

Um número por **epoch**, indicando a condição daquele epoch.



#### O que o classificador tenta fazer?

Ele recebe:

* X (atividade de todos os sensores num instante)
* y (qual condição cada trial representa)

E ele tenta aprender **um padrão espacial** que permita separar Left vs Right.

Isto é:

* “Quando é Left, os sensores têm uma configuração espacial característica.”
* “Quando é Right, a configuração é diferente.”

Esse padrão é chamado de:

* **padrão discriminativo multivariado**
* **multivariate pattern**

Por isso se chama **MVPA = Multivariate Pattern Analysis**.

Ele aprende um **padrão conjunto**, ou seja, usa **todos os sensores ao mesmo tempo**, não cada sensor isolado.



#### Por que “one time point”?

Porque no decoding temporal você treina um modelo **para cada instante do sinal**.

Exemplo:

* Treina um classificador usando os valores no tempo t = 0 ms
* Treina outro para t = 5 ms
* Outro para t = 10 ms
* etc.

Isso gera a famosa curva:

```
tempo → precisão do classificador (%)
```

Ela mostra quando no tempo o cérebro tem informação suficiente para distinguir as condições.


#### Resumo simples

* **X** = matriz com a atividade dos sensores no tempo t para cada epoch.
* **y** = vetor com os rótulos das condições (0 = Left, 1 = Right).
* O classificador aprende a diferença espacial entre as duas condições.
* Ele tenta prever se um novo trial é Left ou Right.


In [ ]:
import numpy as np

# Cria um vetor com comprimento igual ao número de trials.
y = np.empty(len(epochs_auditory.events), dtype=int)

'''
Identifica, para cada trial, se o evento associado é do tipo "Auditory/Left".
epochs_auditory.events é uma matriz Nx3, onde a coluna 2 contém o código do evento.
Compara cada código com o valor correspondente a 'Auditory/Left' no dicionário event_id.

Isso gera um array booleano do mesmo tamanho de y.
Exemplo: idx_left = [False, False, True, True, False, True]
'''
idx_left = epochs_auditory.events[:, 2] == epochs_auditory.event_id['Auditory/Left']

# Faz a mesma coisa para identificar quais trials são "Auditory/Right".
idx_right = epochs_auditory.events[:, 2] == epochs_auditory.event_id['Auditory/Right']

# Codificação das classes: LEFT = 0, RIGHT = 1.

'''
Isso é indexação booleana do NumPy:
- seleciona todos os elementos de y onde idx_left é True
  e atribui o valor 0
- seleciona todos os elementos de y onde idx_right é True
  e atribui o valor 1
'''
y[idx_left] = 0
y[idx_right] = 1

print(idx_left, end="\n\n")

print(y)
print(f'\nTamanho de y: {y.size}')


Agora, vamos criar a matriz de entrada, `X`.

Desejamos focar apenas nos canais de gradiômetro aqui, portanto usamos `pick_types(meg='grad')`. Para canais de magnetômetro, precisaríamos passar `meg='mag'`; e para canais de EEG: `meg=False, eeg=True`.

Criamos uma cópia das epochs porque `pick_types()` opera *in-place*, mas gostaríamos de manter o objeto original de epochs intacto.


In [ ]:
epochs_auditory_grad = epochs_auditory.copy().pick_types(meg='grad')

# Recupera os dados como um array NumPy.
# O array tem o formato: (n_trials, n_channels, n_timepoints)
data = epochs_auditory_grad.get_data()
print(data.shape)  # Número de colunas muito grande

'''
O array retornado é é exatamente uma matriz tridimensional (um tensor 3D) 

Cada epoch contém:
- Vários canais (EEG/MEG)
- Vários instantes no tempo dentro daquela janela temporal

E o conjunto completo de epochs contém:
- Vários epochs diferentes, um para cada trial

Então precisa de 3 eixos para representar tudo.
'''

Estamos quase lá! Precisamos remodelar o array de forma que, para cada ensaio, tenhamos um vetor
`[channel_1_time_1, channel_1_time_2, ..., channel_m_time_n]`, ou seja, nosso objetivo é remodelar `X` para a dimensão `(n_trials, n_channels * n_timepoints)`.

Tinhamos o array:

```
data.shape = (n_trials, n_channels, n_timepoints)
```

Ou seja:

* Para cada trial → uma matriz 2D de sensores × tempo
* O objetivo agora é criar, para **cada trial**, um único vetor grande contendo **todos os sensores concatenados com todos os tempos**.

Isso é necessário porque **classificadores do scikit-learn exigem que X seja 2D**:

```
X.shape = (n_amostras, n_features)
```

No nosso caso:

* **n_amostras = n_trials**
* **n_features = n_channels * n_timepoints**



In [ ]:

'''
Transformar um tensor 3D:

(trials × canais × tempo)

em uma matriz 2D:

(trials × features)

que é o formato necessário para treinar um classificador do scikit-learn.
'''

n_trials = data.shape[0] # Extrai o número de trials (epochs). Se `data` tem shape `(100, 203, 151)`, então `n_trials = 100`.

X = data.reshape(n_trials, -1)
print(X.shape)

'''
-1 = "deixe o NumPy calcular automaticamente essa dimensão".

Suponha que data tenha 6000 elementos no total e que n_trials = 100.
Você faz:
X = data.reshape(100, -1)

O NumPy calcula:
6000 elementos / 100 linhas = 60 colunas


Então:
X.shape → (100, 60)
'''


'''
Então, se antes era:

data.shape = (100, 203, 151)

Após o reshape vira:


X.shape = (100, 203*151)
X.shape = (100, 30653)


Ou seja:

* 100 amostras
* Cada amostra representada por 30 mil “features”

O classificador verá cada trial como um vetor gigante com todos os valores de EEG/MEG concatenados no tempo e nos sensores.
'''



In [ ]:
X # Valores bem pequenos

## Crie um classifier!

Utilizaremos a infraestrutura padrão do `scikit-learn` para a primeira rodada de classificações. O objetivo é demonstrar que é possível simplesmente alimentar o `scikit-learn` com dados pré-processados provenientes do MNE.

Estamos realizando as classificações em **Epochs inteiras** (ou seja, considerando todo o espaço temporal).

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


# --------- Aprendizado Supervisionado --------- #

# Pipeline do classificador: é extremamente importante padronizar os dados
# antes de executar o classificador propriamente dito
# (regressão logística, neste caso).
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

# Executa a validação cruzada.
# CV sem embaralhamento – “block cross-validation” – é o que queremos aqui
# (o scikit-learn não embaralha por padrão, o que é bom para o nosso caso).
n_splits = 5
scoring = 'roc_auc'  # na maioria dos casos é mais robusto do que 'accuracy'
cv = StratifiedKFold(n_splits=n_splits)
scores = cross_val_score(clf, X=X, y=y, cv=cv, scoring=scoring)

# Média e desvio padrão do ROC AUC ao longo das execuções de validação cruzada.
roc_auc_mean = round(np.mean(scores), 3)
roc_auc_std = round(np.std(scores), 3)

print(f'CV scores: {scores}')
print(f'Mean ROC AUC = {roc_auc_mean:.3f} (SD = {roc_auc_std:.3f})')

# Os valores não variam!

### Importações

```python
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
```

Você importa:

* **StandardScaler** → padroniza os dados (muito importante para classificação multivariada)
* **LogisticRegression** → o classificador
* **StratifiedKFold** → cross-validation que mantém a proporção de classes em cada partição
* **cross_val_score** → executa automaticamente o cross-validation
* **make_pipeline** → cria uma pipeline (pré-processamento + modelo)


### Criando o classificador em pipeline

```python
clf = make_pipeline(StandardScaler(),
                    LogisticRegression())
```

A pipeline executa **dois passos** sempre nesta ordem:

1. **StandardScaler()**

   * Calcula média e desvio padrão dos dados **somente no conjunto de treino**
   * Converte cada feature para z-score


2. **LogisticRegression()**

   * Treina um classificador linear para separar as condições (0 = left, 1 = right)
   * Usa regularização interna para não overfitar os milhares de features

**Por que escalação é essencial para EEG/MEG?**

Porque cada feature pode ter escala totalmente diferente
(e.g., sensor A tem 100 fT, sensor B tem 0.1 fT).
Sem escalar, a regressão dá peso só nos sensores de maior amplitude.



### Configurando a Cross-Validation

```python
n_splits = 5
scoring = 'roc_auc'
cv = StratifiedKFold(n_splits=n_splits)
```

#### Explicando cada item:

* **n_splits=5**
  Divide os trials em 5 subconjuntos (5-fold CV).

* **StratifiedKFold**
  Garante que cada fold tem **a mesma proporção** de trials Left e Right.
  Isso é importantíssimo em EEG/MEG para evitar que o modelo treine com 80% Left e 20% Right etc.

* **scoring='roc_auc'**
  Mede o desempenho usando **AUC** e não acurácia.

Por que AUC?
Porque AUC é mais estável quando o dataset é relativamente pequeno.


### Rodando a validação cruzada

```python
scores = cross_val_score(clf, X=X, y=y, cv=cv, scoring=scoring)
```

Isto faz automaticamente:

Para cada fold:

1. Divide X e y em treino/teste
2. Aplica StandardScaler **apenas no treino**, depois no teste
3. Treina a LogisticRegression
4. Calcula o ROC AUC no conjunto de teste
5. Repete para todos os folds

Resultado:

```
scores = [score_fold1, score_fold2, ..., score_fold5]
```

Cada valor é o desempenho do modelo num dos 5 folds. É o AUC obtido em um dos 5 folds da validação cruzada.

Fold 1: 0.9619 → excelente

Fold 2: 0.8143 → ainda bom, mas mais baixo

Fold 3: 0.8667 → bom

Fold 4: 0.8905 → muito bom

Fold 5: 1.0000 → separação perfeita entre as classes nesse fold



### Estatísticas finais

```python
roc_auc_mean = round(np.mean(scores), 3)
roc_auc_std = round(np.std(scores), 3)
```

Calcula:

* **média do AUC** entre os 5 folds
* **desvio padrão**, que indica quão estáveis são os resultados


### RESUMO

Esse bloco:

1. **Cria um classificador linear multivariado**
2. **Escala os dados corretamente**
3. **Faz validação cruzada estratificada**
4. **Avalia usando ROC AUC**
5. **Retorna média e estabilidade do desempenho**

É o pipeline padrão moderno de MVPA em EEG/MEG com scikit-learn.


### Visualize os resultados da validação cruzada


In [ ]:
fig, ax = plt.subplots()
ax.boxplot(scores,
           showmeans=True, # # O triângulo verde indica a média
           whis=(0, 100),  # Whiskers cobrem toda a faixa dos dados
           labels=['Left vs Right'])
ax.set_ylabel('Score')
ax.set_title('Cross-Validation Scores')

## Podemos fazer isso de forma mais simples usando o módulo mne.decoding!

In [ ]:
from sklearn.pipeline import make_pipeline
from mne.decoding import Scaler, Vectorizer, cross_val_multiscore


'''
Exatamente o mesmo tipo de classificação multivariada do exemplo anterior,
mas agora utilizando ferramentas internas do MNE, que automatizam vários passos
e reduzem o risco de erro.
'''

# Primeiro, crie X e y.
epochs_auditory_grad = epochs_auditory.copy().pick_types(meg='grad')
X = epochs_auditory_grad.get_data()
y = epochs_auditory_grad.events[:, 2]

# Pipeline do classificador.
clf = make_pipeline(
    # Um scaler do MNE que lida corretamente com diferentes tipos de canais –
    # isso é ótimo, não é?!
    Scaler(epochs_auditory_grad.info),
    # Converte os dados de (n_epochs, n_channels, n_times)
    # para (n_epochs, n_features)
    Vectorizer(),
    # E, por fim, o classificador propriamente dito.
    LogisticRegression()
)

# Executa a validação cruzada.
# Note que aqui estamos usando o cross_val_multiscore() do MNE,
# e não o cross_val_score() do scikit-learn, como antes.
# Basta passar o número de folds desejados, e o MNE cuida do restante.
n_splits = 5
scoring = 'roc_auc'
scores = cross_val_multiscore(clf, X, y, cv=n_splits, scoring='roc_auc')

# Média e desvio padrão do ROC AUC ao longo das execuções de validação cruzada.
roc_auc_mean = round(np.mean(scores), 3)
roc_auc_std = round(np.std(scores), 3)

print(f'CV scores: {scores}')
print(f'Mean ROC AUC = {roc_auc_mean:.3f} (SD = {roc_auc_std:.3f})')

# Os valores não variam!

## Decodificação ao longo do tempo: comparações em cada ponto temporal individual


Até agora, treinamos um classificador usando todo o trial (toda a época), ou seja, usando todos os sensores ao longo de todos os instantes de tempo juntos como um único vetor de características.

Isso responde a pergunta:

“Essa época pertence à condição A ou à condição B?”

Mas não responde a pergunta mais interessante:

*“Em que momento do tempo o cérebro começa a diferenciar as duas condições?”*

Então, se você usar o sinal completo, você perde a chance de saber quando, especificamente, as condições estão diferentes.


Em vez de pegar:

`[sensores x tempos]`


e transformar tudo em um único vetor, você pega cada instante de tempo separadamente.

Ou seja:

- Tempo 0 ms → treina um classificador só com os valores espaciais (sensores) desse instante
- Tempo 1 ms → outro classificador
- Tempo 2 ms → outro


In [ ]:
from sklearn.preprocessing import StandardScaler
from mne.decoding import SlidingEstimator

# Primeiro, crie X e y.
epochs_auditory_grad = epochs_auditory.copy().pick_types(meg='grad')
X = epochs_auditory_grad.get_data()
y = epochs_auditory_grad.events[:, 2]

# Pipeline do classificador. Não é necessária vetorização como no exemplo anterior.
clf = make_pipeline(StandardScaler(),
                    LogisticRegression())

# O "sliding estimator" irá treinar o classificador em cada ponto no tempo.
scoring = 'roc_auc'
time_decoder = SlidingEstimator(clf, scoring=scoring, n_jobs=1, verbose=True)

# Executa a validação cruzada.
n_splits = 5
scores = cross_val_multiscore(time_decoder, X, y, cv=n_splits, n_jobs=1)

# Média dos scores entre os folds da validação cruzada, para cada ponto no tempo.
mean_scores = np.mean(scores, axis=0)

# Média do score considerando todos os pontos no tempo.
mean_across_all_times = round(np.mean(scores), 3)  # round(valor, ndigits) arredonda o número para a quantidade de casas decimais desejada.
print(f'\n=> Média do score de CV em todos os pontos no tempo: {mean_across_all_times}')


### Imports

```python
from sklearn.preprocessing import StandardScaler
from mne.decoding import SlidingEstimator
```

* **StandardScaler**: normaliza os dados (z-score) antes da classificação.
* **SlidingEstimator**: cria um estimador que treina **um classificador por tempo**, usando apenas o padrão espacial naquele instante.


### Preparando X e y

```python
epochs_auditory_grad = epochs_auditory.copy().pick_types(meg='grad')
X = epochs_auditory_grad.get_data()
y = epochs_auditory_grad.events[:, 2]
```

#### **O que acontece aqui:**

1. **Seleciona apenas sensores gradiômetros (MEG 'grad')**

   ```python
   epochs_auditory.copy().pick_types(meg='grad')
   ```

   Você está escolhendo apenas um tipo de sensor MEG, pois magnetômetros e gradiômetros têm escalas diferentes.

2. **Extrai os dados das epochs**

   ```python
   X = epochs_auditory_grad.get_data()
   ```

   Isso entrega uma matriz com formato:

   ```
   (n_trials, n_sensors, n_times)
   ```

3. **Extrai os rótulos (condições experimentais)**

   ```python
   y = epochs_auditory_grad.events[:, 2]
   ```

   A terceira coluna do array events contém o *código da condição* para cada trial.



### Criando o pipeline do classificador

```python
clf = make_pipeline(StandardScaler(),
                    LogisticRegression())
```

#### **Por que usar pipeline?**

* O MNE automaticamente passa o **padrão espacial dos sensores em cada tempo** para o classificador.
* O StandardScaler garante que todos os sensores fiquem na mesma escala.
* O LogisticRegression é o classificador linear usado para prever a condição.

Isso vira o “classificador base” que será usado pelo SlidingEstimator.



### Criando o SlidingEstimator

```python
scoring = 'roc_auc'
time_decoder = SlidingEstimator(clf, scoring=scoring, n_jobs=1, verbose=True)
```

#### **O que isso faz:**

* **SlidingEstimator** cria **um modelo por tempo**.

* Para cada time point `t`, ele treina:

  ```
  clf.fit(X[:, :, t], y)
  ```

  ou seja, usa **apenas a topografia (sensores) naquele instante**.

* `scoring='roc_auc'` pede AUC como métrica.

* `n_jobs=1` usa apenas 1 core da CPU.

* `verbose=True` imprime progresso.



### Cross-validation

```python
n_splits = 5
scores = cross_val_multiscore(time_decoder, X, y, cv=n_splits, n_jobs=1)
```

#### O que acontece aqui:

* `cross_val_multiscore` roda cross-validation com 5 folds.

* Para cada fold, ele obtém um vetor de scores com tamanho:

  ```
  (n_times,)
  ```

* Resultado final tem formato:

  ```
  scores.shape = (n_splits, n_times)
  ```

Cada linha é um fold separado, cada coluna é um tempo.



### Média dos scores ao longo dos folds

```python
mean_scores = np.mean(scores, axis=0)
```

Isso produz um vetor:

```
(n_times,)
```

Mostrando a curva de performance do classificador ao longo do tempo (time-resolved decoding).



### Média total em todos os tempos

```python
mean_across_all_times = round(np.mean(scores), 3)
print(f'\n=> Mean CV score across all time points: {mean_across_all_times:.3f}')
```

* Faz a média de *todos os tempos e todos os folds*.
* É uma visão global: “em média, o classificador foi X bom ao longo do trial”.


### Resumo

* Extrai dados dos gradiómetros → `X`
* Extrai condições → `y`
* Cria pipeline (escalar + regressão logística)
* Usa SlidingEstimator para treinar um modelo **em cada instante do tempo**
* Executa cross-validation → vetor de scores por tempo
* Calcula médias

### Plotar os resultados da classificação

In [ ]:
fig, ax = plt.subplots()

ax.axhline(0.5, color='k', linestyle='--', label='chance')  # AUC = 0.5
ax.axvline(0, color='k', linestyle='-')  # Mark time point zero.
ax.plot(epochs.times, mean_scores, label='score')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Mean ROC AUC')
ax.legend()
ax.set_title('Left vs Right')
fig.suptitle('Sensor Space Decoding')


# Podemos ver claramente uma excelência da previsão perto de 0,1 segundos